# Introduction

We provide here the solution for the problem of predicting house prices using a MLP based PyTorch implementation.
The configuration is adjusted to run this Notebook on Kaggle.
For runing on Colab or as a Jupyter Notebook locally, adjust the path for the dataset location in the Config dataclass:
```
    data_csv: str = "<your_path_to_dataset>/kc_house_data.csv"
```

# Imports

In [1]:
import copy
import os
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, Dataset

# Reproducibility & Configuration

In [2]:
# =========================================================
# Reproducibility
# =========================================================
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(42)


# =========================================================
# Config
# =========================================================
@dataclass
class Config:
    kaggle: bool = True
    data_csv: str = "/kaggle/input/datasets/harlfoxem/housesalesprediction/kc_house_data.csv"
    submission_csv: str = "submission_pytorch.csv"

    target_col: str = "price"
    id_col: str = "id"

    batch_size: int = 128
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    max_epochs: int = 300
    patience: int = 30

    hidden_dims: tuple = (512, 256, 128)
    dropout: float = 0.15

    valid_size: float = 0.2
    random_state: int = 42

    use_log_target: bool = True


CFG = Config()

# Dataset Class

In [3]:
# =========================================================
# Dataset wrapper
# =========================================================
class TabularRegressionDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray | None = None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = None if y is None else torch.tensor(y, dtype=torch.float32).reshape(-1, 1)

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int):
        if self.y is None:
            return self.X[idx]
        return self.X[idx], self.y[idx]

# Model Class

In [4]:
# =========================================================
# Model
# =========================================================
class HousePriceMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dims=(512, 256, 128), dropout=0.15):
        super().__init__()

        layers = []
        prev_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim

        layers.append(nn.Linear(prev_dim, 1))
        self.model = nn.Sequential(*layers)

        self._init_weights()

    def _init_weights(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
                nn.init.zeros_(module.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)


# Training utilities

In [5]:
# =========================================================
# Training utilities
# =========================================================
def rmse_from_numpy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return np.sqrt(mean_squared_error(y_true, y_pred))


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X_batch.size(0)

    return running_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    preds_all = []
    targets_all = []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        preds = model(X_batch)
        loss = criterion(preds, y_batch)

        running_loss += loss.item() * X_batch.size(0)
        preds_all.append(preds.cpu().numpy())
        targets_all.append(y_batch.cpu().numpy())

    avg_loss = running_loss / len(loader.dataset)
    preds_all = np.vstack(preds_all).ravel()
    targets_all = np.vstack(targets_all).ravel()
    rmse = rmse_from_numpy(targets_all, preds_all)

    return avg_loss, rmse, preds_all, targets_all


@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    preds_all = []

    for X_batch in loader:
        X_batch = X_batch.to(device)
        preds = model(X_batch)
        preds_all.append(preds.cpu().numpy())

    return np.vstack(preds_all).ravel()



# Main training and evaluation pipeline

In [6]:
# =========================================================
# Main pipeline
# =========================================================
def main():
    if not os.path.exists(CFG.data_csv):
        raise FileNotFoundError(f"Could not find {CFG.data_csv}")

    data_df = pd.read_csv(CFG.data_csv)
    train_df, test_df = train_test_split(data_df, test_size=0.2, random_state=42, shuffle=True )

    print(f"Train shape: {train_df.shape}")
    print(f"Test shape:  {test_df.shape}")

    # Preserve IDs for submission
    test_ids = test_df[CFG.id_col].copy()

    # Split features / target
    X = train_df.drop(columns=[CFG.target_col])
    y = train_df[CFG.target_col].copy()
    X_test = test_df.copy()

    # Kaggle score is RMSLE-like, so log-transforming the target helps
    if CFG.use_log_target:
        y = np.log1p(y)

    # Identify column types
    numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
    categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

    # Preprocessing
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )

    # Validation split
    X_train_df, X_valid_df, y_train, y_valid = train_test_split(
        X,
        y,
        test_size=CFG.valid_size,
        random_state=CFG.random_state,
    )

    # Fit preprocessor on train only
    X_train = preprocessor.fit_transform(X_train_df)
    X_valid = preprocessor.transform(X_valid_df)
    X_test_processed = preprocessor.transform(X_test)

    print(f"Processed train shape: {X_train.shape}")
    print(f"Processed valid shape: {X_valid.shape}")
    print(f"Processed test shape:  {X_test_processed.shape}")

    # Datasets / loaders
    train_ds = TabularRegressionDataset(X_train, y_train.values)
    valid_ds = TabularRegressionDataset(X_valid, y_valid.values)
    test_ds = TabularRegressionDataset(X_test_processed)

    train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True)
    valid_loader = DataLoader(valid_ds, batch_size=CFG.batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=CFG.batch_size, shuffle=False)

    # Device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Model
    model = HousePriceMLP(
        input_dim=X_train.shape[1],
        hidden_dims=CFG.hidden_dims,
        dropout=CFG.dropout,
    ).to(device)

    # Optimizer / loss
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=CFG.learning_rate,
        weight_decay=CFG.weight_decay,
    )
    criterion = nn.MSELoss()

    # Training loop with early stopping
    best_model_state = None
    best_valid_loss = float("inf")
    epochs_without_improvement = 0

    for epoch in range(1, CFG.max_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        valid_loss, valid_rmse, _, _ = evaluate(model, valid_loader, criterion, device)

        print(
            f"Epoch {epoch:03d} | "
            f"Train Loss: {train_loss:.6f} | "
            f"Valid Loss: {valid_loss:.6f} | "
            f"Valid RMSE: {valid_rmse:.6f}"
        )

        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= CFG.patience:
            print(f"Early stopping triggered at epoch {epoch}.")
            break

    # Restore best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    # Final validation evaluation
    valid_loss, valid_rmse, valid_preds, valid_targets = evaluate(
        model, valid_loader, criterion, device
    )
    print(f"\nBest validation RMSE (on log target): {valid_rmse:.6f}")

    # Train a final model on full training set if desired
    # For simplicity, here we reuse the model trained on train/valid split.
    # In a competition setting, you may retrain on full train data after choosing hyperparameters.

    # Predict on test
    test_preds = predict(model, test_loader, device)

    # Inverse transform target if needed
    if CFG.use_log_target:
        test_preds = np.expm1(test_preds)

    # Prevent negative prices
    test_preds = np.clip(test_preds, a_min=0, a_max=None)

    submission = pd.DataFrame({
        CFG.id_col: test_ids,
        CFG.target_col: test_preds
    })

    submission.to_csv(CFG.submission_csv, index=False)
    print(f"Submission saved to: {CFG.submission_csv}")
    print(submission.head())

# Run all

In [7]:
if __name__ == "__main__":
    main()

Train shape: (17290, 21)
Test shape:  (4323, 21)
Processed train shape: (13832, 382)
Processed valid shape: (3458, 382)
Processed test shape:  (4323, 382)
Using device: cpu
Epoch 001 | Train Loss: 145.805727 | Valid Loss: 101.920387 | Valid RMSE: 10.095563
Epoch 002 | Train Loss: 55.428299 | Valid Loss: 12.279975 | Valid RMSE: 3.504279
Epoch 003 | Train Loss: 6.533349 | Valid Loss: 0.820593 | Valid RMSE: 0.905866
Epoch 004 | Train Loss: 3.299753 | Valid Loss: 0.337609 | Valid RMSE: 0.581041
Epoch 005 | Train Loss: 2.876441 | Valid Loss: 0.234496 | Valid RMSE: 0.484248
Epoch 006 | Train Loss: 2.551188 | Valid Loss: 0.230809 | Valid RMSE: 0.480426
Epoch 007 | Train Loss: 2.458353 | Valid Loss: 0.300473 | Valid RMSE: 0.548155
Epoch 008 | Train Loss: 2.183695 | Valid Loss: 0.149913 | Valid RMSE: 0.387186
Epoch 009 | Train Loss: 1.892366 | Valid Loss: 0.288663 | Valid RMSE: 0.537274
Epoch 010 | Train Loss: 1.925523 | Valid Loss: 0.123990 | Valid RMSE: 0.352122
Epoch 011 | Train Loss: 1.7032